In [88]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F
import re

from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler

import utils.data_processing_bronze_table
import utils.data_processing_silver_table
import utils.data_processing_gold_table

In [31]:
pd.set_option('display.float_format', '{:.2f}'.format)

In [32]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [93]:
df_attr = spark.read.parquet("datamart/silver/attributes/silver_cust_attr.parquet")

df_attr = df_attr.drop('Name', 'SSN')

# mean_val = df_attr.select(F.mean(col('Age'))).collect()[0][0]
# df_attr = df_attr.fillna({'Age': mean_val})

df_attr = df_attr.fillna({'Occupation': 'Unknown'})


In [94]:
df_attr.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Customer_ID    12500 non-null  object 
 1   Age            12181 non-null  float64
 2   Occupation     12500 non-null  object 
 3   snapshot_date  12500 non-null  object 
dtypes: float64(1), object(3)
memory usage: 390.8+ KB


In [111]:
df_fin.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 34 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Customer_ID                   12500 non-null  object 
 1   Annual_Income                 12500 non-null  float32
 2   Monthly_Inhand_Salary         12500 non-null  float32
 3   Num_Bank_Accounts             12329 non-null  float64
 4   Num_Credit_Card               12204 non-null  float64
 5   Interest_Rate                 12232 non-null  float32
 6   Num_of_Loan                   11933 non-null  float64
 7   Delay_from_due_date           11908 non-null  float64
 8   Num_of_Delayed_Payment        12311 non-null  float64
 9   Changed_Credit_Limit          12500 non-null  float32
 10  Num_Credit_Inquiries          12305 non-null  float64
 11  Credit_Mix                    12500 non-null  object 
 12  Outstanding_Debt              12500 non-null  float32
 13  C

In [114]:
# connect to silver layer
df_fin = spark.read.parquet("datamart/silver/financials/silver_cust_fin.parquet")

# # mean imputation
# mean_cols = ['Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan', 'Num_of_Delayed_Payment']
# for c in mean_cols:
#     mean_val = df_fin.select(F.mean(col(c))).collect()[0][0]
#     df_fin = df_fin.fillna({c: mean_val})

# # median imputation
# median_cols = ['Delay_from_due_date', 'Num_Credit_Inquiries', 'Monthly_Balance']
# for c in median_cols:
#     median_val = df_fin.approxQuantile(c, [0.5], 0.0)[0]
#     df_fin = df_fin.fillna({c: median_val})


    
# fillna(0) — null means no credit change made
df_fin = df_fin.fillna({'Changed_Credit_Limit': 0})

# fillna(unknown) — categorical nulls
df_fin = df_fin.fillna({'Credit_Mix': 'Unknown',
                'Spending_Behaviour': 'Unknown',
                'Payments_Size': 'Unknown'})


# feature engineering
df_fin = df_fin.withColumn('months12_to_annual_income_ratio',
                           (col('Monthly_Inhand_Salary') * 12) / col('Annual_Income')) # detect excessively high annnual income

df_fin = df_fin.withColumn('EMI_to_Salary_Ratio',
                   (col('Total_EMI_per_month') / col('Monthly_Inhand_Salary')))

df_fin = df_fin.withColumn('Debt_to_Income',
                   (col('Outstanding_Debt') / col('Annual_Income')))

df_fin = df_fin.withColumn('Disposable_Income',
                   (col('Monthly_Inhand_Salary') - col('Total_EMI_per_month') - col('Amount_invested_monthly')))



# log transform income-related columns
income_cols = ['Annual_Income', 'Monthly_Inhand_Salary',
               'Outstanding_Debt', 'Total_EMI_per_month',
               'Amount_invested_monthly', 'Monthly_Balance']

for c in income_cols:
    df_fin = df_fin.withColumn(c, F.log1p(col(c)))




In [116]:
df_fin.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 35 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Customer_ID                      12500 non-null  object 
 1   Annual_Income                    12500 non-null  float64
 2   Monthly_Inhand_Salary            12500 non-null  float64
 3   Num_Bank_Accounts                12329 non-null  float64
 4   Num_Credit_Card                  12204 non-null  float64
 5   Interest_Rate                    12232 non-null  float32
 6   Num_of_Loan                      11933 non-null  float64
 7   Delay_from_due_date              11908 non-null  float64
 8   Num_of_Delayed_Payment           12311 non-null  float64
 9   Changed_Credit_Limit             12500 non-null  float32
 10  Num_Credit_Inquiries             12305 non-null  float64
 11  Credit_Mix                       12500 non-null  object 
 12  Outstanding_Debt  

In [117]:
# connect to silver layer
df_loandim = spark.read.parquet("datamart/silver/loan_dimensions/loan_dim.parquet")
df_click = spark.read.parquet("datamart/silver/clickstream/silver_clickstream_daily_*.parquet")

# get customer id and loan_start_date from df_loandim
df_loandim = df_loandim.withColumn("Customer_ID",
                                   F.split(col("loan_id"), r"_(?=\d{4}_\d{2}_\d{2})")[0]).drop("loan_id")

# filter clickstream data for dates before loan start date for each customer
df_click = df_click.join(df_loandim, on="Customer_ID", how="left")
df_click = df_click.filter(col("snapshot_date") < col("loan_start_date"))

# per customer aggregate clickstream data: find daily mean anonymised feature and row count
fe_cols = [c for c in df_click.columns if c.startswith("fe_")]

agg_exprs = [F.mean(col(c)).alias(f"{c}_mean") for c in fe_cols]
agg_exprs.append(F.count("*").alias("clickstream_days"))

df_click = df_click.groupBy("Customer_ID").agg(*agg_exprs)


# JOIN

In [98]:
df_attr.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Customer_ID    12500 non-null  object 
 1   Age            12181 non-null  float64
 2   Occupation     12500 non-null  object 
 3   snapshot_date  12500 non-null  object 
dtypes: float64(1), object(3)
memory usage: 390.8+ KB


In [109]:
df_fin.toPandas()[['Monthly_Inhand_Salary', 'Total_EMI_per_month', 'Amount_invested_monthly']].head()

,Monthly_Inhand_Salary,Total_EMI_per_month,Amount_invested_monthly
0,2706.16,42.94,77.31
1,4250.39,108.37,58.66
2,9549.78,0.00,617.08
3,5208.87,123.43,383.35
4,7962.42,228.02,332.33


In [118]:
df_gold = df_attr.join(df_fin, on=["Customer_ID", "snapshot_date"], how="outer")

click_df_agg = df_click.withColumn("has_clickstream", F.lit(1))

# join click to df
df_gold = df_gold.join(click_df_agg, on="Customer_ID", how="left")

# Customers with no clickstream has_clickstream = null. fill with 0
df_gold = df_gold.fillna({"has_clickstream": 0,
                         "clickstream_days": 0})


In [101]:
df_gold.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 58 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Customer_ID                   12500 non-null  object 
 1   snapshot_date                 12500 non-null  object 
 2   Age                           12181 non-null  float64
 3   Occupation                    12500 non-null  object 
 4   Annual_Income                 12500 non-null  float32
 5   Monthly_Inhand_Salary         12500 non-null  float32
 6   Num_Bank_Accounts             12329 non-null  float64
 7   Num_Credit_Card               12204 non-null  float64
 8   Interest_Rate                 12232 non-null  float32
 9   Num_of_Loan                   11933 non-null  float64
 10  Delay_from_due_date           11908 non-null  float64
 11  Num_of_Delayed_Payment        12311 non-null  float64
 12  Changed_Credit_Limit          12500 non-null  float32
 13  N

# Y data

In [119]:
df_y = spark.read.parquet("datamart/silver/loan_daily/silver_loan_daily_*.parquet")

In [120]:
df_y = df_y.filter(col("mob") == 6)

# get label
df_y = df_y.withColumn("label", F.when(col("dpd") >= 30, 1).otherwise(0).cast(IntegerType()))
df_y = df_y.withColumn("label_def", F.lit(str(30)+'dpd_'+str(6)+'mob').cast(StringType()))

# select columns to save
df_y = df_y.select("loan_id", "Customer_ID", "label", "label_def", "snapshot_date")

In [121]:
df_y.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   loan_id        12500 non-null  object
 1   Customer_ID    12500 non-null  object
 2   label          12500 non-null  int32 
 3   label_def      12500 non-null  object
 4   snapshot_date  12500 non-null  object
dtypes: int32(1), object(4)
memory usage: 439.6+ KB


In [122]:
loan_dim = spark.read.parquet("datamart/silver/loan_dimensions/loan_dim.parquet")

In [123]:
# join x and y
# join loan_start_date back to df_y
df_y = df_y.join(loan_dim, on = 'loan_id', how = 'left')

# rename snapshot_date in df_y to indicate mob6
df_y = df_y.withColumnRenamed('snapshot_date', 'mob6_snapshot_date')

# join df_x and df_y using customer_id and loan start date / snapshot date
df_all = df_gold.join(df_y, 
                   on = [df_gold['Customer_ID'] == df_y['Customer_ID'],
                         df_gold['snapshot_date'] == df_y['loan_start_date']],
                   how = 'inner').drop(df_gold['Customer_ID'], df_y['loan_start_date'])


In [125]:
df_all.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 63 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   snapshot_date                    12500 non-null  object 
 1   Age                              12181 non-null  float64
 2   Occupation                       12500 non-null  object 
 3   Annual_Income                    12500 non-null  float64
 4   Monthly_Inhand_Salary            12500 non-null  float64
 5   Num_Bank_Accounts                12329 non-null  float64
 6   Num_Credit_Card                  12204 non-null  float64
 7   Interest_Rate                    12232 non-null  float32
 8   Num_of_Loan                      11933 non-null  float64
 9   Delay_from_due_date              11908 non-null  float64
 10  Num_of_Delayed_Payment           12311 non-null  float64
 11  Changed_Credit_Limit             12500 non-null  float32
 12  Num_Credit_Inquiri

In [127]:
oot_cutoff = '2024-09-01' # check how many rows there are

df_oot   = df_all.filter(col('snapshot_date') >= oot_cutoff)
df_model = df_all.filter(col('snapshot_date') <  oot_cutoff)

print(f"Model data (train+test): {df_model.count()}")
print(f"OOT data:                {df_oot.count()}")

Model data (train+test): 10022
OOT data:                2478


In [134]:
# convert to pandas for train/test split
df_model_pd = df_model.toPandas()
df_oot_pd   = df_oot.toPandas()


# convert snapshot_date to year and month columns
df_model_pd["snapshot_date"] = pd.to_datetime(df_model_pd["snapshot_date"])
df_model_pd["snapshot_year"]  = df_model_pd["snapshot_date"].dt.year
df_model_pd["snapshot_month"] = df_model_pd["snapshot_date"].dt.month

df_oot_pd["snapshot_date"] = pd.to_datetime(df_oot_pd["snapshot_date"])
df_oot_pd["snapshot_year"]  = df_oot_pd["snapshot_date"].dt.year
df_oot_pd["snapshot_month"] = df_oot_pd["snapshot_date"].dt.month


# define feature and label columns
exclude_cols = ['Customer_ID', 'mob6_snapshot_date', 'loan_id', 
                'label', 'label_def', 'snapshot_date' ] 
feature_cols = [c for c in df_model_pd.columns if c not in exclude_cols]


X = df_model_pd[feature_cols].copy()
y = df_model_pd['label'].copy()

X_oot = df_oot_pd[feature_cols].copy()
y_oot = df_oot_pd['label'].copy()

# train/test split — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 55,
    stratify = y
)

# mean imputation — fit on train only
mean_cols = ['Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan', 
             'Num_of_Delayed_Payment', 'Age']
for c in mean_cols:
    mean_val = X_train[c].mean()                   
    X_train[c] = X_train[c].fillna(mean_val)
    X_test[c]  = X_test[c].fillna(mean_val)        
    X_oot[c]   = X_oot[c].fillna(mean_val)         

# median imputation — fit on train only
median_cols = ['Delay_from_due_date', 'Num_Credit_Inquiries', 
               'Monthly_Balance', 'Interest_Rate']
for c in median_cols:
    median_val = X_train[c].median()               
    X_train[c] = X_train[c].fillna(median_val)
    X_test[c]  = X_test[c].fillna(median_val)      
    X_oot[c]   = X_oot[c].fillna(median_val)       

# clickstream fe_ imputation — fit on train clickers only
fe_cols = [f'fe_{i}_mean' for i in range(1, 21)]
for c in fe_cols:
    mean_val = X_train.loc[X_train['has_clickstream'] == 1, c].mean() 
    X_train[c] = X_train[c].fillna(mean_val)
    X_test[c]  = X_test[c].fillna(mean_val)
    X_oot[c]   = X_oot[c].fillna(mean_val)

# dummy encoding for object columns
cat_cols = ['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 
            'Spending_Behaviour', 'Payments_Size']
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=True)
X_oot   = pd.get_dummies(X_oot,   columns=cat_cols, drop_first=True)


# scaling for 
scaler = StandardScaler()
scale_cols = [
    # continuous numeric — large magnitude differences
    'Age', 'Annual_Income', 'Monthly_Inhand_Salary',
    'Outstanding_Debt', 'Total_EMI_per_month',
    'Amount_invested_monthly', 'Monthly_Balance',
    'Changed_Credit_Limit', 'Credit_Utilization_Ratio',
    'Credit_History_Age_Years', 'snapshot_year', 'snapshot_month',
    
    # count columns — different ranges
    'Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan',
    'Num_Credit_Inquiries', 'Num_of_Delayed_Payment',
    'Delay_from_due_date', 'Interest_Rate', 'clickstream_days',
    
    # engineered features
    'EMI_to_Salary_Ratio', 'Debt_to_Income', 'Disposable_Income',
    'months12_to_annual_income_ratio',

    # loan counts
    'Loan_Auto_Loan', 'Loan_Credit_Builder_Loan',
    'Loan_Debt_Consolidation_Loan', 'Loan_Home_Equity_Loan',
    'Loan_Mortgage_Loan', 'Loan_Not_Specified', 'Loan_Payday_Loan',
    'Loan_Personal_Loan', 'Loan_Student_Loan'
] + [f'fe_{i}_mean' for i in range(1, 21)]

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])
X_oot[scale_cols]   = scaler.transform(X_oot[scale_cols])



print(f"Train: {len(X_train)} rows, bad rate: {y_train.mean():.1%}")
print(f"Test:  {len(X_test)}  rows, bad rate: {y_test.mean():.1%}")
print(f"OOT:   {len(X_oot)}   rows, bad rate: {y_oot.mean():.1%}")

Train: 8017 rows, bad rate: 29.0%
Test:  2005  rows, bad rate: 29.0%
OOT:   2478   rows, bad rate: 28.0%


In [135]:
X_train.describe()

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,...,fe_15_mean,fe_16_mean,fe_17_mean,fe_18_mean,fe_19_mean,fe_20_mean,clickstream_days,has_clickstream,snapshot_year,snapshot_month
count,8017.00,8017.00,8017.00,8017.00,8017.00,8017.00,8017.00,8017.00,8017.00,8017.00,...,8017.00,8017.00,8017.00,8017.00,8017.00,8017.00,8017.00,8017.00,8017.00,8017.00
mean,0.00,-0.00,0.00,-0.00,-0.00,-0.00,0.00,-0.00,0.00,0.00,...,-0.00,0.00,0.00,-0.00,-0.00,0.00,0.00,0.84,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.36,1.00,1.00
min,-1.84,-1.81,-2.83,-2.07,-2.70,-1.55,-1.47,-1.56,-2.19,-2.41,...,-7.58,-6.87,-6.96,-6.85,-8.00,-8.38,-1.36,0.00,-0.83,-1.45
25%,-0.82,-0.75,-0.82,-0.91,-0.76,-0.75,-0.64,-0.76,-0.72,-0.75,...,-0.46,-0.46,-0.47,-0.47,-0.47,-0.47,-1.00,1.00,-0.83,-0.83
50%,0.00,-0.06,-0.00,-0.00,0.00,-0.18,-0.22,-0.20,0.09,-0.14,...,0.00,0.00,0.00,0.00,0.00,0.00,-0.10,1.00,-0.83,-0.20
75%,0.77,0.63,0.81,0.63,0.70,0.62,0.61,0.52,0.74,0.66,...,0.45,0.45,0.46,0.46,0.48,0.45,0.97,1.00,1.21,0.75
max,2.08,6.69,1.97,2.18,2.65,9.78,2.28,2.84,2.37,3.87,...,6.60,6.40,7.69,6.72,7.07,7.70,1.68,1.00,1.21,2.00
